In [5]:
import os
import pandas as pd
import pdfplumber

In [6]:
def extract_tables_from_pdf(file_path, output_dir, target_pages):
    os.makedirs(output_dir, exist_ok=True)

    with pdfplumber.open(file_path) as pdf:
        for page_index, file_name in target_pages.items():
            full_path = os.path.join(output_dir, file_name)
            
            # Check if the file already exists
            if os.path.exists(full_path):
                print(f"Skipping: {file_name} already exists in {output_dir}")
                continue

            page = pdf.pages[page_index]
            table = page.extract_table()
            
            if table:
                df = pd.DataFrame(table[1:], columns=table[0])

                # Clean column names and data
                df.columns = [str(c).replace('\n', ' ') if c else f"Unnamed_{i}" for i, c in enumerate(df.columns)]
                df = df.replace('\n', ' ', regex=True)
                
                # Save to the directory
                df.to_csv(full_path, index=False)
                print(f"Successfully saved: {full_path}")
            else:
                print(f"Warning: No table found on page {page_index + 1}")

In [7]:
file_path = "data/original_data/participation_data/2024-sfia-topline.pdf"
output_dir = os.path.join("data", "extracted_data")
target_pages = {
    31: "SFIA_Aerobic_Activities.csv",
    32: "SFIA_Conditioning_Activities.csv",
    33: "SFIA_Team_Sports.csv"
}

extract_tables_from_pdf(file_path, output_dir, target_pages)

Skipping: SFIA_Aerobic_Activities.csv already exists in data\extracted_data
Skipping: SFIA_Conditioning_Activities.csv already exists in data\extracted_data
Skipping: SFIA_Team_Sports.csv already exists in data\extracted_data


In [8]:
def filter_by_code(main_table, mapping_table, target_cols_list, code_col_name="Code"):
    valid_codes = mapping_table[code_col_name].values

    mask = main_table[target_cols_list].isin(valid_codes).any(axis=1)

    return main_table[mask]

In [9]:
main_table = pd.read_excel('data/original_data/injury_data/NEISS_2024.XLSX')
mapping_table = pd.read_csv('data/original_data/codes/NEISS_tables/product_codes.csv')

filtered_table = filter_by_code(main_table, mapping_table, ['Product_1', 'Product_2', 'Product_3'])

print(main_table.shape)
print(filtered_table.shape)

(361672, 25)
(98267, 25)


In [ ]:
def replace_code_with_meaning(main_table, mapping_table, target_cols, meaning_col, code_col="Code"):
    # Ensure new object will be returned
    new_table = main_table.copy()
    mapping_dict = dict(zip(mapping_table[code_col], mapping_table[meaning_col]))

    # Apply the mapping to all target columns
    for col in target_cols:
        new_table[col] = new_table[col].map(mapping_dict).fillna(new_table[col])
    
    return new_table

In [ ]:
replaced_data_table = replace_code_with_meaning(
    filtered_table, 
    mapping_table, 
    ['Product_1', 'Product_2', 'Product_3'].copy(),
    meaning_col="Sport"
)

print(replaced_data_table.head())

    CPSC_Case_Number Treatment_Date  Age  Sex  Race Other_Race  Hispanic  \
0          240108461     2024-01-01   16    1     0        NaN         1   
2          240109863     2024-01-01   10    1     1        NaN         2   
11         240109877     2024-01-01    6    2     1        NaN         2   
13         240110364     2024-01-01   15    1     1        NaN         2   
22         240111495     2024-01-01   32    1     4        NaN         2   

    Body_Part  Diagnosis  Other_Diagnosis  ...  Fire_Involvement  Alcohol  \
0          30         64              NaN  ...                 0        0   
2          93         54              NaN  ...                 0        0   
11         80         57              NaN  ...                 0        0   
13         37         64              NaN  ...                 0        0   
22         30         71  SHOULDER INJURY  ...                 0        0   

   Drug                                     Product_1  Product_2  Product_3  \
0